# Stage 1 Unattended Kaggle Benchmark: Coimbra 4Q (`MER_musicality`)

Runs the complete **Tier 1 (4 ch) -> Tier 2 (6 ch) -> Tier 3 (23 ch)** cognitive feature ladder and **10-fold stratified benchmark** (`Summary-LogReg`, `Summary-RBF-SVM`, `CogMucs-TCN`, `CogMucs-GRU`, plus Leave-One-Branch-Out ablations).

### Kaggle Setup Checklist (Before Clicking Save & Run All)
1. **Accelerator:** Choose `GPU T4 x2` or `GPU P100` in the right sidebar.
2. **Internet:** Set to `ON` (required to clone your GitHub repo).
3. **Add Input Dataset:** Click **Add Input** and attach **`4q-audio-emotion-dataset-russell`** (`sparshg/4q-audio-emotion-dataset-russell`).
4. **Run Unattended:** Click **Save Version -> Save & Run All (Commit)**. You can safely turn off your PC once the commit starts; all CSVs, plots, and checkpoints will be saved in the Kaggle Output tab.

In [ ]:
import os, subprocess, sys, torch
from pathlib import Path

print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

REPO_URL = 'https://github.com/ssoravitt/MER_musicality.git'
REPO_DIR = Path('/kaggle/working/MER_musicality')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = str(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Install only lightweight missing packages if needed (librosa, torch, scipy, sklearn are usually preinstalled)
for pkg in ('librosa', 'soundfile', 'yaml'):
    try:
        __import__(pkg)
    except ImportError:
        pip_name = 'pyyaml' if pkg == 'yaml' else pkg
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

print('Working dir:', Path.cwd())

In [ ]:
!python -m pytest tests/test_cogmucs.py -q

In [ ]:
import os, subprocess, sys
from collections import Counter
from data.coimbra4q import auto_discover_kaggle_4q, discover_coimbra4q

data_root = auto_discover_kaggle_4q('/kaggle/input')
assert data_root is not None, 'Could not find Q1..Q4 directories under /kaggle/input. Please attach 4q-audio-emotion-dataset-russell.'
print('Located Coimbra 4Q root at:', data_root)

records = discover_coimbra4q(data_root)
print('Discovered tracks:', len(records), '| Counts per class idx:', Counter(r.label for r in records))

PROCESSED_ROOT = '/kaggle/working/processed'
SPLITS_CSV = '/kaggle/working/splits/coimbra4q_splits.csv'
WORKERS = min(4, os.cpu_count() or 2)

cmd = [
    sys.executable, '-m', 'scripts.extract',
    '--dataset', 'coimbra4q',
    '--tier', '3',
    '--target-hz', '10',
    '--workers', str(WORKERS),
    '--data-dir', str(data_root),
    '--out-dir', PROCESSED_ROOT,
    '--splits-out', SPLITS_CSV,
]
subprocess.run(cmd, check=True)

In [ ]:
import os, subprocess, sys

OUT_DIR = '/kaggle/working/results'
cmd = [
    sys.executable, '-m', 'scripts.benchmark_coimbra4q',
    '--processed-dir', '/kaggle/working/processed/coimbra4q/tier3_10hz',
    '--splits-csv', '/kaggle/working/splits/coimbra4q_splits.csv',
    '--out-dir', OUT_DIR,
    '--n-folds', '10',
    '--epochs', '60',
    '--batch-size', '32',
]
subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

res_dir = Path('/kaggle/working/results')
display(Markdown((res_dir / 'summary_report.md').read_text(encoding='utf-8')))

for img_name in ('tier_ablation_plot.png', 'branch_weights_by_quadrant.png', 'confusion_matrices.png'):
    img_path = res_dir / img_name
    if img_path.exists():
        display(Image(filename=str(img_path)))